In [1]:
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

from ontologx.backend.tests import TestsFactory

load_dotenv("../.env")

NEO4J_URL = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "password"  # noqa: S105
TESTS_FILE = "../resources/data/test.ttl"
SHACL_PATH = "../resources/ontologies/logs_shacl.ttl"

store = Neo4jGraph(
    url=NEO4J_URL,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

# Load the tests evaluator
tests_evaluator = TestsFactory.create(
    backend="bedrock",
    model="us.meta.llama3-3-70b-instruct-v1:0",
)

In [2]:
import neo4j
from langchain_core.documents import Document

from ontologx.store import GraphDocument, Node, Relationship


def get_subgraph_from_node(node_uri: str, props_to_remove: list[str] | None = None) -> GraphDocument:
    """Get the subgraph of a node in the store.

    The subgraph will contain all the nodes and relationships connected to the given node, even indirectly.
    """
    if props_to_remove is None:
        props_to_remove = []

    props_to_remove = [*props_to_remove, "runName", "embedding"]

    # Ugly but quite efficient. Also filters out the embedding property and the Resource label.
    nodes_subgraphs = store.query(
        """
        MATCH (n {uri: $node_uri})
        CALL apoc.path.subgraphAll(n, {labelFilter: '-Dataset'})
        YIELD nodes, relationships
        RETURN
        [node IN nodes | {
        uri: node.uri,
        type: HEAD([label IN LABELS(node) WHERE label <> 'Resource']),
        properties: apoc.map.removeKeys(PROPERTIES(node), $props_to_remove)
        }] AS nodes,
        [rel IN relationships | {
        source: STARTNODE(rel).uri,
        target: ENDNODE(rel).uri,
        type: TYPE(rel)
        }] AS relationships
        """,
        params={"node_uri": node_uri, "props_to_remove": props_to_remove},
    )

    if not nodes_subgraphs:
        return GraphDocument(nodes=[], relationships=[])

    nodes_subgraph = nodes_subgraphs[0]

    # The neo4j date and time objects are quite problematic, as they are not JSON serializable.
    # This is a workaround to convert them to strings.
    for node in nodes_subgraph["nodes"]:
        for key, value in node["properties"].items():
            if isinstance(value, neo4j.time.DateTime | neo4j.time.Date | neo4j.time.Time):
                node["properties"][key] = value.iso_format()

    def get_node_id(uri: str) -> str:
        """Get the node id from the node uri."""
        final_str = uri.split("/")[-1]

        if "#" in final_str:
            return final_str.split("#")[-1]

        return final_str

    nodes_dict = {
        node["uri"]: Node(id=get_node_id(node["uri"]), type=node["type"], properties=node["properties"])
        for node in nodes_subgraph["nodes"]
    }

    relationships = (
        [
            Relationship(
                source=nodes_dict[relationship["source"]],
                target=nodes_dict[relationship["target"]],
                type=relationship["type"],
            )
            for relationship in nodes_subgraph["relationships"]
        ]
        if "relationships" in nodes_subgraph
        else []  # The node may not have any relationships
    )

    return GraphDocument(
        nodes=list(nodes_dict.values()),
        relationships=relationships,
    )


def tests() -> list[GraphDocument]:
    test_nodes = store.query(
        """
        MATCH (d:Dataset)-[:hasPart]->(e:Event)
        WHERE d.type = "tests"
        RETURN e.eventMessage as message, e.uri as uri
        ORDER BY e.uri
        """,
    )

    tests = []
    for test in test_nodes:
        graph = get_subgraph_from_node(test["uri"])

        source_node = next((node for node in graph.nodes if node.type == "Source"), None)
        context = (
            {
                "sourceName": source_node.properties["sourceName"],
                "sourceDevice": source_node.properties["sourceDevice"],
            }
            if source_node
            else {}
        )
        graph.source = Document(page_content=test["message"], metadata=context)
        tests.append(graph)

    return tests

In [3]:
from pathlib import Path

store.query(
    "CALL n10s.validation.shacl.import.inline($constraints, 'Turtle')",
    params={"constraints": Path(SHACL_PATH).read_text()},
)

[]

In [4]:
# Get all generated graphs
from ontologx import accuracy

# Get all runs
run_uris = store.query(
    """
    MATCH (r:Run)
    RETURN r.uri as uri
    """,
)
print(len(run_uris), "runs found")

all_metrics = []

for run_uri in run_uris:
    y_true = tests()
    y_pred = []
    n_shacl_violations = 0
    for graph in y_true:
        # Get the event node
        event_node = next((node for node in graph.nodes if node.type == "Event"), None)
        if event_node is None:
            msg = "No Event node found in the graph"
            raise ValueError(msg)

        # Get the corresponding pred graph
        pred_event_node_uri = store.query(
            """
            MATCH (r:Run)-[:hasOutput]->(d:Dataset)-[:hasPart]->(e:Event)
            WHERE e.eventMessage = $event_message AND r.uri = $run_uri
            RETURN e.uri as uri
            """,
            params={
                "event_message": event_node.properties["eventMessage"],
                "run_uri": run_uri["uri"],
            },
        )
        if len(pred_event_node_uri) == 0 or pred_event_node_uri[0]["uri"] is None:
            y_pred.append(GraphDocument(nodes=[], relationships=[]))
            continue

        check_shacl = store.query(
            """
            MATCH (n {uri: $node_uri})
            CALL apoc.path.subgraphAll(n, {labelFilter: '-Dataset'})
            YIELD nodes
            CALL n10s.validation.shacl.validateSet(nodes)
            YIELD focusNode, nodeType,propertyShape,offendingValue,resultPath,severity
            RETURN focusNode, nodeType,propertyShape,offendingValue,resultPath,severity
            """,
            params={"node_uri": pred_event_node_uri[0]["uri"]},
        )
        if len(check_shacl) > 0:
            pred_graph = GraphDocument(nodes=[], relationships=[])
            n_shacl_violations += 1
        else:
            pred_graph = get_subgraph_from_node(pred_event_node_uri[0]["uri"])
            pred_graph.source = graph.source

        y_pred.append(pred_graph)

    metrics = accuracy.AccuracyEvaluator(y_pred, y_true, tests_evaluator)

    results = {
        "SHACL_violations_percentage": n_shacl_violations / len(y_pred),
        "f1_score": metrics.f1(),
        "alignment": metrics.alignment(),
        "BERT score": metrics.bert_score(),
    }
    all_metrics.append(results)


/mnt/servicesdata/lcotti/conda/envs/lcotti-conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


11 runs found
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!
False !!!!!!!!!!!!


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
import numpy as np

metrics_names = all_metrics[0].keys()
mean_std_metrics = {}

for metric in metrics_names:
    values = [m[metric] for m in all_metrics]
    mean_std_metrics[metric] = {
        "mean": np.mean(values),
        "std_dev": np.std(values),
    }

for metric, stats in mean_std_metrics.items():
    print(f"{metric}: mean={stats['mean']}, std_dev={stats['std_dev']}")

SHACL_violations_percentage: mean=0.0, std_dev=0.0
f1_score: mean=0.6999096645399692, std_dev=0.022158930848573902
alignment: mean=0.5567272727272727, std_dev=0.01485132935029973
BERT score: mean=0.6261797005479987, std_dev=0.00928657964466703
